# 9. 地震可以預測嗎？從預測走向機率預報

第一部的結尾（{doc}`第 8 章 <08_explore_ideas>`）留下了一個誠實但
不太舒服的結論：想在單一
觀測量裡找到可靠的地震前兆，非常困難；前兆研究的歷史是一座墓園。
那麼，科學就對地震束手無策了嗎？

不是。過去三十年，地震科學走通了另一條路：放棄「預測下一場大地震
的時間地點」這個目標，改為回答一個謙虛得多、但答得出來的問題——

> **在這個區域、這段時間內，發生某規模以上地震的機率是多少？**

這條路叫做**地震預報（earthquake forecasting）**。它的材料不是神祕
的前兆訊號，而是你在{doc}`第 5 章 <05_seismic>`已經摸過的東西：
地震目錄本身。第二部
要帶你把這條路完整走一遍——從目錄統計的進階課，到 ETAS、EEPAS
這些真正在世界各地上線運轉的模型，到「預報做得好不好」的檢驗
科學，最後回到台灣。

本章先把最重要的觀念釘牢：預測與預報差在哪、我們憑什麼能預報、
以及一份「地震預報」實際上長什麼樣子。

## 9.1 預測與預報：一字之差

中文的「預測」與「預報」日常可以互換，但在地震科學裡，這兩個詞
被刻意分開，而且分得很嚴格：

| | 地震預測（prediction） | 地震預報（forecast） |
|---|---|---|
| 形式 | 確定性：「某時某地將發生規模 X 地震」 | 機率性：「此區此期間發生 M≥X 的機率為 p」 |
| 範圍 | 窄到可以據此疏散（Main 1999） | 一個區域、一段時間窗、一個規模區間 |
| 現況 | **做不到**，且短期內看不到希望 | **已經在多國作業化運轉** |

確定性預測的失敗不是沒試過。美國在 Parkfield 斷層段守了十幾年，
預測的地震遲到了十年才來；各種前兆方案（第 8 章那座墓園）在事後
檢驗中一一倒下。台灣中央氣象署自己的業務回顧也給出誠實的數字：
短期前兆觀測中「可視為成功發現前兆的比例」在**兩成以下**
（蕭乃祺，約 2019）。

這裡還要先拆掉一個常見的混淆。台灣民眾最熟悉的「地震警報」——
手機在搖晃前幾秒響起——是**地震預警（early warning）**：地震
**已經發生**，只是破壞性的震波還在路上，搶的是波速與電磁波速之間
的幾秒到幾十秒。台灣的預警是世界前段班（2018 年花蓮地震後 17 秒
發布、20 秒觸達手機）。而**預報**問的是「地震還沒發生時，接下來
發生的機率」——時間尺度從幾天到幾十年。這兩件事的物理、方法、
難度完全不同：台灣在預警上領先全球，在預報上則和所有國家一樣，
只能給機率。

那機率預報憑什麼給得出來？憑地震目錄裡最強、最穩定的一個訊號。

## 9.2 叢集：目錄裡最強的訊號

地震在時間上不是均勻隨機的。如果地震像理想的隨機過程（Poisson
過程）那樣互不相關地發生，事件之間的等待時間應該呈指數分布。
拿 2024 年春天的目錄（第 5 章用過的那份，含 0403 花蓮主震與
餘震序列）來檢查：

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import ndimage

from gdms_toolkit.download import CACHE_DIR
from gdms_toolkit.viz import ACCENT, QUAKE_COLOR, apply_layout

cat = pd.read_csv(CACHE_DIR / "catalog_2024spring.csv", parse_dates=["time"])
dt_hours = cat.time.sort_values().diff().dt.total_seconds().dropna() / 3600
mean_dt = dt_hours.mean()

edges = np.arange(0, 12.25, 0.25)
hist, _ = np.histogram(dt_hours, bins=edges, density=True)
centers = edges[:-1] + 0.125

fig = go.Figure()
fig.add_trace(go.Bar(x=centers, y=hist, name="觀測（2024 春季目錄）",
                     marker_color=ACCENT, opacity=0.75))
fig.add_trace(go.Scatter(x=centers, y=np.exp(-centers / mean_dt) / mean_dt,
                         mode="lines", name="同平均率的 Poisson 過程",
                         line=dict(color="#eda100", width=2.5)))
apply_layout(fig, title="事件間隔時間分布：地震不是隨機撒點",
             xaxis_title="與前一事件的間隔（小時）",
             yaxis_title="機率密度", yaxis_type="log", hovermode="x")
fig

觀測分布（藍）在短間隔端遠高於同平均發生率的 Poisson 預期（黃線）：
大量地震擠在前一個地震後的幾分鐘到一小時內。這就是**叢集
（clustering）**——地震會觸發地震，一次大震後餘震成串而來，
你在第 5 章的 M–T 圖上已經親眼看過。

叢集有兩條百年經驗律撐腰，都是你在第 5 章算過的：

- **Omori–Utsu 律**：餘震發生率隨主震後時間衰減，
  $n(t) = \dfrac{K}{(t+c)^p}$，$p$ 典型接近 1；
- **Gutenberg–Richter 律**：$\log_{10} N = a - bM$，$b$ 典型接近 1，
  規模每小一級、數量約多十倍。

把兩條合起來，就能回答「接下來這一週，這附近發生 M≥5 地震的機率」
——Omori 給「還會有多少餘震」，GR 給「其中多大比例會是大的」。
**現代地震預報的核心引擎，本質上就是這兩條定律的各種精緻組合。**
這也解釋了一件重要的事：現行預報模型最擅長的是叢集（餘震），
而不是憑空冒出的大地震。

這裡藏著一個反直覺、但對防災至關重要的觀念：**前震、主震、餘震
只是事後標籤**。目前沒有任何已知的物理量能在當下區分「這是餘震」
還是「這是更大地震的前震」——序列還在進行時，永遠無法排除更大的
還在後面。統計上這不是杞人憂天：1980–2019 年間，全球 M≥6 地震
有超過一成在 60 天、100 公里內被**更大**的地震跟上（Taroni 2023）。2016 年熊本地震是最痛的一課：M6.5 發生後，日本氣象廳
依慣例發布「餘震」預報，28 小時後真正的主震 M7.3 才來。
此後 JMA 認定原有程序失效，修改了整套發布方式。

所以，機率預報不只是「算得出來」，還是**唯一誠實的說法**：
序列進行中，科學能說的就是「接下來一週再來一個 M≥6 的機率是
百分之幾」，而不是「餘震會慢慢變小，請放心」。

## 9.3 預報長什麼樣：從機率地圖到作業化系統

一份現代的地震預報，具體長什麼樣子？它通常是一張**網格化的機率
地圖**：把區域切成格子，模型對每個格子給出「單位時間內發生
目標規模以上地震的期望數（或機率）」。用春季目錄做一個最簡單的
示意——把觀測到的地震活動度在空間上平滑，就得到一張「哪裡
比較容易再有地震」的地圖雛形：

In [ ]:
step = 0.1
lon_edges = np.arange(119.0, 123.5 + step, step)
lat_edges = np.arange(21.0, 26.0 + step, step)
counts, _, _ = np.histogram2d(cat.latitude, cat.longitude,
                              bins=[lat_edges, lon_edges])
weeks = (cat.time.max() - cat.time.min()).days / 7
rate = ndimage.gaussian_filter(counts, sigma=1.2) / weeks  # 每格每週期望數

main = cat.loc[cat.ML.idxmax()]
fig = go.Figure(go.Heatmap(
    x=lon_edges[:-1] + step / 2, y=lat_edges[:-1] + step / 2,
    z=np.log10(rate + 1e-4), colorscale="Blues",
    colorbar=dict(title="log₁₀ 期望數<br>（每格每週）")))
fig.add_trace(go.Scatter(x=[main.longitude], y=[main.latitude],
                         mode="markers", name="0403 花蓮主震",
                         marker=dict(symbol="x", size=12, color=QUAKE_COLOR,
                                     line=dict(width=2))))
apply_layout(fig, title="平滑後的地震活動度地圖（2024 春季，ML≥3）",
             xaxis_title="經度", yaxis_title="緯度", hovermode="closest",
             yaxis_scaleanchor="x", height=560)
fig

真正的預報模型當然比「把過去抹平」聰明得多——它們會建模觸發
（第 13 章）、前兆尺度（第 16 章）、與長期背景（第 12、16 章），
而且機率會隨每一個新地震即時更新。但輸出的形式就是這樣一張
隨時間演化的機率地圖。順帶一提，「把過去的地震活動平滑成未來的
預期」這個最樸素的想法，本身就是一個正式的模型家族（PPE，
第 16 章），而且是所有花俏模型都必須贏過的基準線。

當這樣的機率地圖被**自動化、定期產出，並由權責機關發布**，就叫做
**作業化地震預報（Operational Earthquake Forecasting, OEF）**。
這個詞是 2009 年義大利 L'Aquila 地震（Mw 6.3，約 300 人罹難）之後，
國際地震預報委員會（ICEF）給出的方向（Jordan et al. 2011）：
與其在「能不能預測」上空轉，不如把**當下已知的機率**用權威、
透明、常態化的方式交到社會手上。定義裡有兩個關鍵字：

- **作業化（operational）**：借自氣象學——自動、及時、持續運轉，
  不是研究者事後發表論文；
- **權威性（authoritative）**：資訊來自依法負有職責的機關。權威性
  不能自我宣稱，必須由制度授予——這句話把 OEF 從純技術問題變成
  制度問題。

世界上目前有三種代表性的做法：

| | 義大利 INGV | 紐西蘭 GNS/GeoNet | 美國 USGS |
|---|---|---|---|
| 模型 | ETAS／ETES／STEP 集成 | 短期 STEP/ETAS＋中期 EEPAS＋長期 PPE 混成 | Reasenberg–Jones 貝氏／ETAS |
| 發布 | 每日更新，未來一週 | 隨序列演化調整 | 大震後 20 分鐘自動發布 |
| 公開程度 | **不對大眾公開**（僅民防體系） | 完全公開 | 完全公開 |

三國都能算機率，分歧在「要不要、怎麼給社會」——再一次，瓶頸
不在統計。機率的量級感也值得先建立：2016 年紐西蘭 Kaikōura
M7.8 之後兩週，GeoNet 發布的未來 7 天機率是——M5.0–5.9 至少
一個：**98%**；M6.0–6.9**：41%**；M≥7**：5%**——換算成期望
個數是 5.6、0.53、0.05 個，同一份預報裡跨了兩個數量級。「5% 的 M≥7」該不該讓城市停擺？
這種「低機率、高衝擊」的溝通難題，是 OEF 至今最硬的挑戰，
我們會在第 22 章回來。

## 9.4 CSEP：把模型放上擂台

讀到這裡你應該會想問：模型百百種，怎麼知道誰的機率值得信？

第 8 章的 8.2 節其實已經給過答案的雛形：**規則要在看到答案之前
定好**（第 3 關）**、要用沒有地震的時段檢驗誤報率**（第 4 關）。
把這兩條紀律放大成國際建制，就是 **CSEP**（Collaboratory for the
Study of Earthquake Predictability，地震可預測性研究合作計畫）：

- 模型開發者把模型**預先**提交到獨立的測試中心；
- 模型對未來（不是過去！）的地震給出正式預報；
- 幾年後，用**事先議定**的統計檢驗為所有模型打分數。

這叫**前瞻性檢驗（prospective testing）**，它一刀切掉了第 8 章
講的事後選擇偏誤：你不可能對還沒發生的地震 p-hacking。國際
專家社群的共識講得很直白：「模型開發者自己相信自己的模型」
**不構成任何證據**；模型換一個地區使用，必須重新檢驗。CSEP
二十年來累積的另一個教訓同樣重要：不少模型在回溯測試中大放
異彩，一到前瞻測試就被打回原形（第 14、15 章會看到具體案例）。

第二部的每一章講到任何模型，我們都會問同一個問題：**它前瞻
檢驗過了嗎？成績如何**？這是這個領域跟前兆墓園最大的差別——
它建立了淘汰機制。

## 9.5 第二部的地圖

接下來八章的路線是這樣的：

| 章 | 主題 | 時間尺度 |
|---|---|---|
| 10 | 目錄統計進階：Mc、b 值、除叢——所有模型的地基 | — |
| 11 | ETAS：把「地震觸發地震」寫成一條式子 | 天～週 |
| 12 | EEPAS 與 PPE：前兆尺度增加與中長期預報 | 月～十年 |
| 13 | STEP 與作業化系統：模型如何上線 | 天～週 |
| 14 | 模型組合：混合與加乘 | — |
| 15 | 預報檢驗：CSEP 的工具箱 | — |
| 16 | PSHA：從預報到危害度 | 數十年 |
| 17 | 台灣：資料、模型與下一步 | — |

一個提醒：第二部以**觀念**為主。每章的圖都是用簡單模擬或真實
目錄產生的示意圖，程式碼預設摺疊——你不需要會實作這些模型，
但讀完之後，你應該能看懂任何一篇地震預報論文在做什麼、並且
知道該對它問哪些問題。

最後，把第一部的精神帶過來。第 8 章教你對「看起來像前兆的異常」
保持懷疑；第二部要教你的，是對「看起來很準的模型」保持同樣的
懷疑——只是這一次，懷疑有了正式的工具：基準模型、前瞻檢驗、
資訊增益。地震預報這門學問最了不起的地方，不在於它能算出機率，
而在於它建立了一套讓錯誤的機率無所遁形的制度。
{doc}`下一章 <11_catalog_completeness_b>`，我們先回到一切的地基：
地震目錄，以及那些比第 5 章更深的坑。